# Model Selection and Decision Threshold Optimization

This notebook extends the model development and hyperparameter optimization conducted in `03_model_training.ipynb`.

Out-of-fold (OOF) predicted probabilities are generated for the tuned logistic regression, BERT, and RoBERTa classifiers using five-fold stratified cross-validation. These probabilities are subsequently used to evaluate alternative classification thresholds and compare model performance across sensitivity, specificity, precision, F1-score, and balanced accuracy.

The objective is to identify a model–threshold combination that balances the identification of survivor narratives with the accurate exclusion of non-narrative posts. The selected configuration will subsequently be evaluated using the independent, previously held-out test dataset.

*By the end of Notebook 04, we should be able to answer three questions.*

*First: Which classifier offers the best balance between identifying survivor narratives and excluding irrelevant posts?*

- *We will compare tuned weighted logistic regression, BERT, and RoBERTa using balanced accuracy, sensitivity, specificity, precision, and F1.*

*Second: Is the conventional 0.50 decision threshold appropriate?*

- *Perhaps BERT performs best at 0.45, while RoBERTa performs best at 0.60. We can determine which threshold optimizes our prespecified selection criterion.*

*Third: Which model should we use to classify the remaining unannotated Reddit posts?*

- *This is arguably the most important practical outcome. Your manually annotated sample contains 580 posts, but your cleaned corpus contains 2,960 posts.*

*We ultimately want a classifier capable of identifying survivor narratives across that larger corpus so you can conduct the subsequent dissertation analyses.*

In [1]:

import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [2]:

PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = Path.cwd() / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Results directory:", RESULTS_DIR)


Project directory: /Users/Kashif/Library/CloudStorage/Box-Box/0.KB_NLP
Results directory: /Users/Kashif/Library/CloudStorage/Box-Box/0.KB_NLP/notebooks/results


In [3]:

train_df = pd.read_csv(
    DATA_DIR / "narr_train_70.csv"
)

X_train = (
    train_df["title"].fillna("") + " " +
    train_df["selftext"].fillna("")
).str.strip()

y_train = train_df["survivor_narrative"].astype(int)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Training observations:", len(X_train))
print("\nClass distribution:")
print(y_train.value_counts())


Training observations: 406

Class distribution:
survivor_narrative
1    306
0    100
Name: count, dtype: int64


## Out-of-Fold Predictions

### Tuned Logistic Regression

The tuned, class-weighted logistic regression classifier is evaluated using five-fold stratified cross-validation. Predicted probabilities are generated for each held-out validation fold, ensuring that each training observation receives a prediction from a model that was not trained on that observation.

The resulting OOF probabilities will be used for subsequent decision threshold optimization.

In [4]:

# Reconstruct the tuned logistic regression pipeline.
tuned_lr = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 1),
        min_df=2
    )),
    ("classifier", LogisticRegression(
        C=1,
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

# Generate out-of-fold predicted probabilities.
lr_oof_prob = cross_val_predict(
    tuned_lr,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# Identify the validation fold for each observation.
lr_oof_folds = np.zeros(len(y_train), dtype=int)

for fold, (_, val_idx) in enumerate(
    cv.split(X_train, y_train), start=1
):
    lr_oof_folds[val_idx] = fold

# Save OOF predictions.
lr_oof_df = pd.DataFrame({
    "observation_id": np.arange(len(y_train)),
    "fold": lr_oof_folds,
    "true_label": y_train.to_numpy(),
    "predicted_probability": lr_oof_prob
})

lr_oof_df.to_csv(
    RESULTS_DIR / "lr_tuned_oof_predictions.csv",
    index=False
)

print(f"Saved {len(lr_oof_df)} OOF predictions.")
display(lr_oof_df.head())


Saved 406 OOF predictions.


/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:451: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


,observation_id,fold,true_label,predicted_probability
0,0,3,1,0.723753
1,1,3,1,0.657581
2,2,2,1,0.805598
3,3,4,1,0.598284
4,4,1,0,0.380938


In [5]:
assert len(lr_oof_df) == 406
assert lr_oof_df["observation_id"].is_unique
assert lr_oof_df["fold"].between(1, 5).all()
assert lr_oof_df["predicted_probability"].between(0, 1).all()

print("All OOF validation checks passed.")

All OOF validation checks passed.



### Tuned BERT

Out-of-fold predicted probabilities were generated for the tuned BERT classifier using five-fold stratified cross-validation. The previously selected hyperparameters were held constant across folds.

For each fold, a new BERT model was fine-tuned using four-fifths of the training observations and used to predict the held-out fifth. The resulting probabilities were retained for decision threshold optimization and comparison with logistic regression and RoBERTa.


In [8]:

# Prepare the training data for BERT.
MODEL_NAME = "bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

bert_df = pd.DataFrame({
    "text": X_train.to_numpy(),
    "labels": y_train.to_numpy().astype(int)
})

bert_dataset = Dataset.from_pandas(
    bert_df,
    preserve_index=False
)

# Tokenize using the same maximum length as model training.
def tokenize_bert(batch):
    return bert_tokenizer(
        batch["text"],
        truncation=True,
        max_length=512
    )

bert_tokenized = bert_dataset.map(
    tokenize_bert,
    batched=True,
    remove_columns=["text"]
)

bert_data_collator = DataCollatorWithPadding(
    tokenizer=bert_tokenizer
)

# Configure the selected BERT hyperparameters.
bert_args = TrainingArguments(
    output_dir=str(RESULTS_DIR / "bert_oof"),
    learning_rate=3e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
    seed=42
)

# Store OOF probabilities in their original observation order.
bert_oof_prob = np.zeros(len(y_train))
bert_oof_folds = np.zeros(len(y_train), dtype=int)

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train), start=1
):
    print(f"\n--- Tuned BERT Fold {fold}/5 ---", flush=True)

    torch.manual_seed(42)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    trainer = Trainer(
        model=model,
        args=bert_args,
        train_dataset=bert_tokenized.select(train_idx.tolist()),
        eval_dataset=bert_tokenized.select(val_idx.tolist()),
        data_collator=bert_data_collator
    )

    trainer.train()

    # Predict probabilities for the held-out validation fold.
    predictions = trainer.predict(
        bert_tokenized.select(val_idx.tolist())
    )

    probabilities = torch.softmax(
        torch.tensor(predictions.predictions),
        dim=1
    )[:, 1].numpy()

    bert_oof_prob[val_idx] = probabilities
    bert_oof_folds[val_idx] = fold

    # Save completed predictions after each fold.
    bert_oof_df = pd.DataFrame({
        "observation_id": np.arange(len(y_train)),
        "fold": bert_oof_folds,
        "true_label": y_train.to_numpy(),
        "predicted_probability": bert_oof_prob
    })

    bert_oof_df.to_csv(
        RESULTS_DIR / "bert_tuned_oof_predictions.csv",
        index=False
    )

    print(f"Fold {fold} complete.", flush=True)

    del trainer, model
    gc.collect()

    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

print("\nBERT OOF prediction generation complete.")
display(bert_oof_df.head())


Map:   0%|          | 0/406 [00:00<?, ? examples/s]


--- Tuned BERT Fold 1/5 ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/opt/anaconda3/lib/pyt

Epoch,Training Loss,Validation Loss
1,0.644662,0.309242
2,0.392717,0.332548
3,0.112053,0.371027


Fold 1 complete.

--- Tuned BERT Fold 2/5 ---


... (13 outputs left out)

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.648422,0.165031
2,0.356761,0.412856
3,0.160904,0.248155


Fold 5 complete.

BERT OOF prediction generation complete.


,observation_id,fold,true_label,predicted_probability
0,0,3,1,0.999515
1,1,3,1,0.999336
2,2,2,1,0.998865
3,3,4,1,0.998338
4,4,1,0,0.002402


In [9]:
assert len(bert_oof_df) == 406
assert bert_oof_df["observation_id"].is_unique
assert bert_oof_df["fold"].between(1, 5).all()
assert bert_oof_df["predicted_probability"].between(0, 1).all()

print("All BERT OOF validation checks passed.")

All BERT OOF validation checks passed.



### Tuned RoBERTa: Out-of-Fold Predictions

Using the optimal hyperparameters identified during model tuning, RoBERTa was retrained across the same five stratified folds used for logistic regression and BERT.

For each fold, the model was trained on four subsets of the training data and generated predicted probabilities for observations in the held-out fifth subset. This procedure produced one out-of-fold predicted probability for each of the 406 training observations.

The resulting probabilities were saved for subsequent decision threshold optimization and model comparison.


In [10]:

# Prepare the training data for RoBERTa.
ROBERTA_MODEL = "roberta-base"

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL)

def tokenize_roberta(batch):
    return roberta_tokenizer(
        batch["text"],
        truncation=True,
        max_length=512
    )

roberta_tokenized = bert_dataset.map(
    tokenize_roberta,
    batched=True,
    remove_columns=["text"]
)

roberta_data_collator = DataCollatorWithPadding(
    tokenizer=roberta_tokenizer
)

# Configure the selected RoBERTa hyperparameters.
roberta_args = TrainingArguments(
    output_dir=str(RESULTS_DIR / "roberta_oof"),
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
    seed=42
)

# Store OOF probabilities in their original observation order.
roberta_oof_prob = np.full(len(y_train), np.nan)
roberta_oof_folds = np.zeros(len(y_train), dtype=int)

for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_train, y_train), start=1
):
    print(f"\n--- Tuned RoBERTa Fold {fold}/5 ---", flush=True)

    torch.manual_seed(42)

    model = AutoModelForSequenceClassification.from_pretrained(
        ROBERTA_MODEL,
        num_labels=2
    )

    trainer = Trainer(
        model=model,
        args=roberta_args,
        train_dataset=roberta_tokenized.select(train_idx.tolist()),
        eval_dataset=roberta_tokenized.select(val_idx.tolist()),
        data_collator=roberta_data_collator
    )

    trainer.train()

    # Predict probabilities for the held-out validation fold.
    predictions = trainer.predict(
        roberta_tokenized.select(val_idx.tolist())
    )

    probabilities = torch.softmax(
        torch.tensor(predictions.predictions),
        dim=1
    )[:, 1].numpy()

    roberta_oof_prob[val_idx] = probabilities
    roberta_oof_folds[val_idx] = fold

    # Save completed predictions after each fold.
    roberta_oof_df = pd.DataFrame({
        "observation_id": np.arange(len(y_train)),
        "fold": roberta_oof_folds,
        "true_label": y_train.to_numpy(),
        "predicted_probability": roberta_oof_prob
    })

    roberta_oof_df.to_csv(
        RESULTS_DIR / "roberta_tuned_oof_predictions.csv",
        index=False
    )

    print(f"Fold {fold} complete.", flush=True)

    del trainer, model
    gc.collect()

    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

print("\nRoBERTa OOF prediction generation complete.")
display(roberta_oof_df.head())


Map:   0%|          | 0/406 [00:00<?, ? examples/s]


--- Tuned RoBERTa Fold 1/5 ---


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  su

Epoch,Training Loss,Validation Loss
1,0.780203,0.482827
2,0.399451,0.319312
3,0.152254,0.327983


Fold 1 complete.

--- Tuned RoBERTa Fold 2/5 ---


... (13 outputs left out)

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.700653,0.587804
2,0.486954,0.328544
3,0.170731,0.284094


Fold 5 complete.

RoBERTa OOF prediction generation complete.


,observation_id,fold,true_label,predicted_probability
0,0,3,1,0.999456
1,1,3,1,0.999549
2,2,2,1,0.999171
3,3,4,1,0.999256
4,4,1,0,0.003757


In [11]:

assert len(roberta_oof_df) == 406
assert roberta_oof_df["observation_id"].is_unique
assert roberta_oof_df["fold"].between(1, 5).all()
assert roberta_oof_df["predicted_probability"].notna().all()
assert roberta_oof_df["predicted_probability"].between(0, 1).all()

print("All RoBERTa OOF validation checks passed.")


All RoBERTa OOF validation checks passed.



## Loading and Validating Out-of-Fold Predictions

Out-of-fold (OOF) predicted probabilities from the tuned weighted logistic regression, BERT, and RoBERTa classifiers were loaded for model comparison and classification threshold optimization.

Each classifier generated one predicted probability for each of the 406 observations in the training dataset using five-fold stratified cross-validation. Each observation was predicted by a model trained without that observation.

Before proceeding, the prediction files were checked for completeness, unique observation identifiers, valid probability values, and consistency of manually annotated labels across classifiers.

The independent test dataset (n = 174) was excluded from this stage.


In [ ]:

# Load OOF predictions from all three tuned classifiers

lr_oof = pd.read_csv(
    RESULTS_DIR / "lr_tuned_oof_predictions.csv"
)

bert_oof = pd.read_csv(
    RESULTS_DIR / "bert_tuned_oof_predictions.csv"
)

roberta_oof = pd.read_csv(
    RESULTS_DIR / "roberta_tuned_oof_predictions.csv"
)

oof_models = {
    "Logistic Regression": lr_oof,
    "BERT": bert_oof,
    "RoBERTa": roberta_oof
}

# Validate each model's predictions
for model_name, df in oof_models.items():

    assert len(df) == 406, f"{model_name}: incorrect number of observations"

    assert df["observation_id"].is_unique, f"{model_name}: duplicate observations"

    assert df["predicted_probability"].notna().all(), f"{model_name}: missing probabilities"

    assert df["predicted_probability"].between(0, 1).all(), f"{model_name}: invalid probabilities"

    assert df["true_label"].isin([0, 1]).all(), f"{model_name}: invalid labels"

    print(f"{model_name}: {len(df)} valid OOF predictions")

# Confirm all models predicted the same observations with the same labels
reference = lr_oof.set_index("observation_id")["true_label"].sort_index()

for model_name, df in oof_models.items():

    labels = df.set_index("observation_id")["true_label"].sort_index()

    assert labels.equals(reference), f"{model_name}: observation or label mismatch"

print("\nAll OOF prediction files are aligned and validated.")



### Examining Out-of-Fold Predicted Probabilities

The distribution of out-of-fold predicted probabilities was examined separately for manually annotated survivor narratives (1) and non-survivor posts (0) across all three classifiers.

Descriptive statistics were calculated to assess how each classifier assigned probabilities to the two outcome classes. Ideally, survivor narratives should receive higher predicted probabilities, while non-survivor posts should receive lower probabilities.

These distributions provide an initial assessment of class separation before classification threshold optimization.


In [13]:

# Summarize OOF predicted probabilities by model and true label

probability_summaries = []

for model_name, df in oof_models.items():

    summary = (
        df.groupby("true_label")["predicted_probability"]
        .agg(["count", "mean", "median", "std", "min", "max"])
        .reset_index()
    )

    summary.insert(0, "model", model_name)

    probability_summaries.append(summary)

probability_summary_df = pd.concat(
    probability_summaries,
    ignore_index=True
)

display(probability_summary_df.round(3))


,model,true_label,count,mean,median,std,min,max
0,Logistic Regression,0,100,0.393,0.389,0.126,0.162,0.758
1,Logistic Regression,1,306,0.687,0.705,0.133,0.251,0.931
2,BERT,0,100,0.192,0.002,0.376,0.001,1.000
3,BERT,1,306,0.945,0.999,0.221,0.002,1.000
4,RoBERTa,0,100,0.171,0.008,0.346,0.001,0.999
5,RoBERTa,1,306,0.943,0.999,0.214,0.002,1.000



### Classification Threshold Evaluation

Classification performance was evaluated across thresholds ranging from 0.05 to 0.95 in increments of 0.01 using out-of-fold predicted probabilities.

At each threshold, predicted probabilities were converted into binary classifications and compared against manually annotated labels. Sensitivity, specificity, precision, F1-score, accuracy, and balanced accuracy were calculated.

Balanced accuracy, defined as the average of sensitivity and specificity, was designated as the primary threshold-selection criterion to account for class imbalance in the annotated training dataset.

Thresholds were evaluated separately for weighted logistic regression, BERT, and RoBERTa. The independent test dataset was not used during threshold selection.


In [14]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)

threshold_results = []

thresholds = np.round(np.arange(0.05, 0.951, 0.01), 2)

for model_name, df in oof_models.items():

    y_true = df["true_label"].to_numpy()
    y_prob = df["predicted_probability"].to_numpy()

    for threshold in thresholds:

        y_pred = (y_prob >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_true, y_pred, labels=[0, 1]
        ).ravel()

        threshold_results.append({
            "model": model_name,
            "threshold": threshold,
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "sensitivity": recall_score(y_true, y_pred),
            "specificity": tn / (tn + fp),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "f1": f1_score(y_true, y_pred, zero_division=0)
        })

threshold_results_df = pd.DataFrame(threshold_results)

print("Threshold evaluations complete.")
print(f"Total evaluations: {len(threshold_results_df)}")

display(threshold_results_df.head())


Threshold evaluations complete.
Total evaluations: 273


,model,threshold,accuracy,precision,sensitivity,specificity,balanced_accuracy,f1
0,Logistic Regression,0.05,0.753695,0.753695,1.0,0.0,0.5,0.859551
1,Logistic Regression,0.06,0.753695,0.753695,1.0,0.0,0.5,0.859551
2,Logistic Regression,0.07,0.753695,0.753695,1.0,0.0,0.5,0.859551
3,Logistic Regression,0.08,0.753695,0.753695,1.0,0.0,0.5,0.859551
4,Logistic Regression,0.09,0.753695,0.753695,1.0,0.0,0.5,0.859551



### Selection of Optimal Classification Thresholds

For each classifier, the threshold maximizing out-of-fold balanced accuracy was identified from the evaluated threshold range (0.05–0.95).

When multiple thresholds produced identical maximum balanced accuracy, the threshold closest to the conventional 0.50 cutoff was selected.

The resulting thresholds were compared across classifiers using sensitivity, specificity, precision, F1-score, accuracy, and balanced accuracy. These results were used to inform model selection before independent test-set evaluation.


In [15]:

# Identify the threshold maximizing balanced accuracy for each model

optimal_thresholds = []

for model_name, df in threshold_results_df.groupby("model"):

    max_balanced_accuracy = df["balanced_accuracy"].max()

    candidates = df[
        np.isclose(df["balanced_accuracy"], max_balanced_accuracy)
    ].copy()

    # Tie-breaker: choose the threshold closest to 0.50
    candidates["distance_from_050"] = (
        candidates["threshold"] - 0.50
    ).abs()

    best = candidates.sort_values(
        ["distance_from_050", "threshold"]
    ).iloc[0]

    optimal_thresholds.append(best.drop("distance_from_050"))

optimal_thresholds_df = (
    pd.DataFrame(optimal_thresholds)
    .sort_values("balanced_accuracy", ascending=False)
    .reset_index(drop=True)
)

display(optimal_thresholds_df.round(3))


,model,threshold,accuracy,precision,sensitivity,specificity,balanced_accuracy,f1
0,RoBERTa,0.93,0.916,0.963,0.925,0.89,0.907,0.943
1,BERT,0.95,0.921,0.954,0.941,0.86,0.901,0.947
2,Logistic Regression,0.53,0.869,0.963,0.859,0.90,0.880,0.908



### Comparison of Default and Optimized Classification Thresholds

Classification performance at the conventional threshold of 0.50 was compared with performance at each classifier's balanced-accuracy-optimized threshold.

This comparison assessed whether threshold optimization improved balanced accuracy and how changes in the decision threshold affected sensitivity, specificity, precision, and F1-score.

The comparison was conducted using training-set out-of-fold predictions. Final model performance will be evaluated separately using the independent test dataset.


In [16]:

# Compare performance at default and optimized thresholds

default_results = threshold_results_df[
    np.isclose(threshold_results_df["threshold"], 0.50)
].copy()

default_results["threshold_type"] = "Default"

optimized_results = optimal_thresholds_df.copy()
optimized_results["threshold_type"] = "Optimized"

comparison_df = pd.concat(
    [default_results, optimized_results],
    ignore_index=True
)

comparison_df = comparison_df[
    [
        "model",
        "threshold_type",
        "threshold",
        "sensitivity",
        "specificity",
        "balanced_accuracy",
        "precision",
        "f1",
        "accuracy"
    ]
].sort_values(["model", "threshold"])

display(comparison_df.round(3))


,model,threshold_type,threshold,sensitivity,specificity,balanced_accuracy,precision,f1,accuracy
1,BERT,Default,0.50,0.944,0.81,0.877,0.938,0.941,0.911
4,BERT,Optimized,0.95,0.941,0.86,0.901,0.954,0.947,0.921
0,Logistic Regression,Default,0.50,0.902,0.84,0.871,0.945,0.923,0.887
5,Logistic Regression,Optimized,0.53,0.859,0.90,0.880,0.963,0.908,0.869
2,RoBERTa,Default,0.50,0.948,0.84,0.894,0.948,0.948,0.921
3,RoBERTa,Optimized,0.93,0.925,0.89,0.907,0.963,0.943,0.916



## Sensitivity-Prioritized Classification Threshold Selection

Given the dissertation's objective of constructing an inclusive corpus of male sexual violence survivor narratives, a sensitivity-prioritized threshold-selection strategy was adopted.

Thresholds achieving sensitivity of at least 0.95 were identified for each classifier. Among qualifying thresholds, specificity was maximized to limit the inclusion of non-survivor posts while maintaining high sensitivity.

This approach prioritizes the identification of genuine survivor narratives, including potentially ambiguous accounts, while recognizing the possibility of false-positive classifications. The sensitivity target was selected as an exploratory screening criterion and will require independent evaluation.

The independent test dataset remained excluded from threshold selection.


In [17]:

# Select thresholds prioritizing sensitivity

MIN_SENSITIVITY = 0.95

sensitivity_results = []

for model_name, df in threshold_results_df.groupby("model"):

    eligible = df[
        df["sensitivity"] >= MIN_SENSITIVITY
    ].copy()

    if eligible.empty:
        print(f"{model_name}: No threshold meets sensitivity target.")
        continue

    # Maximize specificity while maintaining sensitivity >= 0.95
    best = eligible.sort_values(
        ["specificity", "balanced_accuracy", "threshold"],
        ascending=[False, False, False]
    ).iloc[0]

    sensitivity_results.append(best)

sensitivity_optimal_df = (
    pd.DataFrame(sensitivity_results)
    .sort_values(
        ["specificity", "sensitivity"],
        ascending=False
    )
    .reset_index(drop=True)
)

display(sensitivity_optimal_df.round(3))


,model,threshold,accuracy,precision,sensitivity,specificity,balanced_accuracy,f1
0,RoBERTa,0.32,0.921,0.945,0.951,0.83,0.890,0.948
1,BERT,0.15,0.914,0.936,0.951,0.80,0.875,0.943
2,Logistic Regression,0.46,0.892,0.909,0.951,0.71,0.830,0.930



# Error Analysis of the Selected RoBERTa Classifier

An exploratory error analysis was conducted using out-of-fold predictions from the tuned RoBERTa classifier at the provisionally selected sensitivity-prioritized threshold of 0.32.

Predicted classifications were compared with manually annotated survivor-narrative labels to identify false negatives (genuine survivor narratives incorrectly excluded) and false positives (non-survivor posts incorrectly included).

Particular attention was given to false negatives because the dissertation aims to retain potentially ambiguous, indirect, or uncertain accounts of sexual victimization.

This analysis was conducted using training-set out-of-fold predictions and did not involve the independent test dataset.


In [18]:

# Identify RoBERTa classification errors at threshold 0.32

SELECTED_THRESHOLD = 0.32

roberta_errors_df = roberta_oof.copy()

# Convert probabilities into binary classifications
roberta_errors_df["predicted_label"] = (
    roberta_errors_df["predicted_probability"] >= SELECTED_THRESHOLD
).astype(int)

# Identify classification outcomes
roberta_errors_df["classification_outcome"] = np.select(
    [
        (roberta_errors_df["true_label"] == 1) &
        (roberta_errors_df["predicted_label"] == 1),

        (roberta_errors_df["true_label"] == 0) &
        (roberta_errors_df["predicted_label"] == 0),

        (roberta_errors_df["true_label"] == 0) &
        (roberta_errors_df["predicted_label"] == 1),

        (roberta_errors_df["true_label"] == 1) &
        (roberta_errors_df["predicted_label"] == 0)
    ],
    [
        "True Positive",
        "True Negative",
        "False Positive",
        "False Negative"
    ],
    default="Unknown"
)

# Summarize classification outcomes
error_summary = (
    roberta_errors_df["classification_outcome"]
    .value_counts()
    .reindex([
        "True Positive",
        "True Negative",
        "False Positive",
        "False Negative"
    ], fill_value=0)
)

display(error_summary.to_frame(name="count"))

# Separate misclassified observations
false_negatives_df = roberta_errors_df[
    roberta_errors_df["classification_outcome"] == "False Negative"
].copy()

false_positives_df = roberta_errors_df[
    roberta_errors_df["classification_outcome"] == "False Positive"
].copy()

print(f"\nFalse negatives: {len(false_negatives_df)}")
print(f"False positives: {len(false_positives_df)}")


,count
classification_outcome,
True Positive,291
True Negative,83
False Positive,17
False Negative,15



False negatives: 15
False positives: 17



## Retrieval of Misclassified Posts for Qualitative Error Analysis

Misclassified observations were identified using their original observation identifiers and linked to the corresponding Reddit post text in the annotated training dataset.

False negatives were prioritized for qualitative review to investigate whether incorrectly excluded survivor narratives shared characteristics such as indirect disclosure, uncertainty about victimization, or ambiguous descriptions of sexual violence.

False positives were also retained for subsequent review to identify patterns contributing to the incorrect inclusion of non-survivor posts.

The analysis was exploratory and intended to characterize classification errors rather than modify the trained classifier or optimize its threshold further.


In [19]:

# Retrieve original text for misclassified observations

# Confirm the text aligns with the OOF observation indices
assert len(X_train) == len(roberta_oof)

# Create a lookup table
post_text_lookup = pd.DataFrame({
    "observation_id": np.arange(len(X_train)),
    "post_text": X_train.to_numpy()
})

# Attach original text to false negatives
false_negatives_text_df = false_negatives_df.merge(
    post_text_lookup,
    on="observation_id",
    how="left",
    validate="one_to_one"
)

# Attach original text to false positives
false_positives_text_df = false_positives_df.merge(
    post_text_lookup,
    on="observation_id",
    how="left",
    validate="one_to_one"
)

# Confirm all posts were successfully retrieved
assert false_negatives_text_df["post_text"].notna().all()
assert false_positives_text_df["post_text"].notna().all()

print("False-negative posts retrieved:", len(false_negatives_text_df))
print("False-positive posts retrieved:", len(false_positives_text_df))

# Preview false negatives, ordered by predicted probability
display(
    false_negatives_text_df[
        ["observation_id", "predicted_probability", "post_text"]
    ]
    .sort_values("predicted_probability")
    .head(5)
)


False-negative posts retrieved: 15
False-positive posts retrieved: 17


,observation_id,predicted_probability,post_text
13,342,0.001828,Perverts in this sub Asked for help and get th...
4,108,0.002036,Question I have a question. How did you rememb...
11,325,0.002242,History of Abuse and Hypersexuality Not even s...
9,239,0.003262,imposter syndrome being abused by my sister do...
7,165,0.003613,Leaving for adult survivor therapy summer camp...



### Qualitative Examination of False-Negative Narratives

The full text of false-negative observations was reviewed to identify characteristics potentially associated with incorrect exclusion from the survivor-narrative corpus.

Initial examination prioritized observations assigned the lowest survivor-narrative probabilities by RoBERTa. The review considered narrative structure, explicitness of victimization disclosure, contextual references, and potential ambiguity in the original manual annotations.

Error characteristics were documented inductively, without assuming that all misclassifications reflected the same underlying mechanism.


In [21]:

# Display the complete text of all 15 false negatives

all_fn = (
    false_negatives_text_df
    .sort_values("predicted_probability")
)

for _, row in all_fn.iterrows():

    print("=" * 80)
    print(f"Observation ID: {row['observation_id']}")
    print(f"Predicted probability: {row['predicted_probability']:.4f}")
    print(f"Manual label: {row['true_label']}")
    print("-" * 80)
    print(row["post_text"])
    print("\n")


Observation ID: 342
Predicted probability: 0.0018
Manual label: 1
--------------------------------------------------------------------------------
Perverts in this sub Asked for help and get three dms in an hour asking about details of what happened, and one guy even sent me a dick pic. wtf?


Observation ID: 108
Predicted probability: 0.0020
Manual label: 1
--------------------------------------------------------------------------------
Question I have a question. How did you remember that you were abused? This is a legit question. Looking for answers on my own memory of things.


Observation ID: 325
Predicted probability: 0.0022
Manual label: 1
--------------------------------------------------------------------------------
History of Abuse and Hypersexuality Not even sure where to begin.

It started at 6. Continued at 11. More at 14 and 16.

Different situations, all intense and helped shape a pattern of hypersexuality and years of therapy.

I recognize there are people here who lik

In [22]:

# Display the complete text of all 17 false positives

all_fp = (
    false_positives_text_df
    .sort_values("predicted_probability", ascending=False)
)

for _, row in all_fp.iterrows():

    print("=" * 80)
    print(f"Observation ID: {row['observation_id']}")
    print(f"Predicted probability: {row['predicted_probability']:.4f}")
    print(f"Manual label: {row['true_label']}")
    print("-" * 80)
    print(row["post_text"])
    print("\n")


Observation ID: 59
Predicted probability: 0.9994
Manual label: 0
--------------------------------------------------------------------------------
question Lately I've been questioning the validity of my memories. I know something happened, but I may never know what. I remember, though, that my mother told me once when I was a small child I'd said I had a dream my brother touched me. But my brother is a year and a half younger. He would've probably been like 2, maybe 3. Does anyone know of this happening? Like, if babies will explore another child without knowing what they're doing? I always assumed it must've been my father, but then why wouldn't I have said I dreamed my father did it?


Observation ID: 148
Predicted probability: 0.9993
Manual label: 0
--------------------------------------------------------------------------------
Why Is This Happening I’ve never had sex with anyone. Never touched anyone intimately. At least, never when I was awake. And yet, I keep getting vivid hypno


## Final Model Selection

### Selection Rationale

Three classification algorithms were evaluated for identifying first-person survivor narratives: TF-IDF with class-weighted logistic regression, BERT, and RoBERTa. Model development included stratified five-fold cross-validation, hyperparameter tuning, and out-of-fold probability estimation.

Because the primary objective was to identify eligible survivor narratives for subsequent dissertation analyses, model selection prioritized sensitivity to minimize the exclusion of potentially relevant accounts. Candidate decision thresholds were evaluated using out-of-fold predictions, with a target sensitivity of at least 0.95 and specificity used to distinguish models meeting that criterion.

RoBERTa (`roberta-base`) was selected with a decision threshold of **0.32**. At this threshold, out-of-fold performance was:

| Metric | Value |
|---|---:|
| Accuracy | 0.921 |
| Precision | 0.945 |
| Sensitivity | 0.951 |
| Specificity | 0.830 |
| Balanced accuracy | 0.890 |
| F1-score | 0.948 |

### Error Analysis

Of the 406 out-of-fold predictions, 374 agreed with the manually assigned labels, while 32 disagreed (15 false negatives and 17 false positives).

Exploratory review suggested that classification challenges included indirect disclosures, uncertainty about victimization, narratives emphasizing recovery or help-seeking, and posts with limited experiential detail. Several disagreements also raised questions about the consistency of reference annotations.

These findings illustrate the interpretive complexity of identifying survivor narratives and the limitations of treating human-assigned labels as error-free. The error analysis was exploratory and was not used to revise the selected threshold.

### Final Selection and Next Steps

**Selected model:** RoBERTa (`roberta-base`)

**Decision threshold:** 0.32

**Training configuration:**
- Learning rate: 1e-5
- Epochs: 3
- Batch size: 2
- Weight decay: 0.01
- Maximum sequence length: 512

The selected model and threshold will be evaluated on the independent held-out test set (n = 174), which was not used for model development or selection.

Out-of-fold performance should be interpreted as a model-development estimate rather than an independent assessment of generalization. The independent test evaluation will provide additional evidence regarding the classifier's suitability for identifying survivor narratives.

The broader methodological objective is not error-free classification, but a transparent, reproducible, and scalable approach to extending human-annotated judgments to a larger corpus.


In [28]:

import json
from pathlib import Path

# Selected model configuration
selected_model_config = {
    "model_name": "RoBERTa",
    "pretrained_model": "roberta-base",
    "decision_threshold": 0.32,
    "hyperparameters": {
        "learning_rate": 1e-5,
        "num_train_epochs": 3,
        "per_device_train_batch_size": 2,
        "weight_decay": 0.01,
        "max_length": 512,
        "random_state": 42
    },
    "selection_criteria": {
        "primary_metric": "sensitivity",
        "minimum_sensitivity": 0.95,
        "secondary_metric": "specificity",
        "selection_data": "5-fold out-of-fold training predictions"
    },
    "training_sample_size": 406,
    "held_out_test_size": 174
}

# Save configuration to project models directory
MODELS_DIR = Path("../models/survivor_narrative_roberta").resolve()
MODELS_DIR.mkdir(parents=True, exist_ok=True)

config_path = MODELS_DIR / "survivor_narrative_roberta_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(selected_model_config, f, indent=4)

print(f"Configuration saved to: {config_path}")
print(f"File exists: {config_path.exists()}")


Configuration saved to: /Users/Kashif/Library/CloudStorage/Box-Box/0.KB_NLP/models/survivor_narrative_roberta/survivor_narrative_roberta_config.json
File exists: True
